# Olist KPI Framework - Ədalət Sadıqov

## Libraries

In [1]:
import oracledb
print(oracledb.__version__)

26.0.1


In [2]:
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from getpass import getpass
from sqlalchemy.engine import URL
import seaborn as sns

sns.set_theme(style="whitegrid")

In [3]:
db_password = getpass("Oracle password: ")

engine = create_engine(
    URL.create(
        drivername="oracle+oracledb",
        username="Brazilian_olist",
        password=db_password,
        host="localhost",
        port=1521,
        query={"service_name": "XEPDB1"},
    )
)

In [4]:
test = pd.read_sql("SELECT COUNT(*) AS row_count FROM olist_customers", engine)
test

,row_count
0,99441


## Questions

### q1

In [5]:
q1_rows = """
SELECT 'name_translation' AS table_name, COUNT(*) AS row_count FROM name_translation
UNION ALL SELECT 'olist_customers', COUNT(*) FROM olist_customers
UNION ALL SELECT 'olist_geolocation', COUNT(*) FROM olist_geolocation
UNION ALL SELECT 'olist_orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'olist_order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'olist_order_payments', COUNT(*) FROM olist_order_payments
UNION ALL SELECT 'olist_order_reviews', COUNT(*) FROM olist_order_reviews
UNION ALL SELECT 'olist_products', COUNT(*) FROM olist_products
UNION ALL SELECT 'olist_sellers', COUNT(*) FROM olist_sellers"""

In [6]:
q1_rows_result = pd.read_sql(q1_rows, engine)
q1_rows_result

,table_name,row_count
0,name_translation,71
1,olist_customers,99441
2,olist_geolocation,1000163
3,olist_orders,99441
4,olist_order_items,112650
5,olist_order_payments,103886
6,olist_order_reviews,121648
7,olist_products,32951
8,olist_sellers,3095


In [7]:
q1_card = """
WITH order_item_counts AS (
    SELECT order_id, COUNT(*) AS items_per_order
    FROM olist_order_items
    GROUP BY order_id
)
SELECT COUNT(*)                        AS orders_with_items,
       ROUND(AVG(items_per_order), 3)  AS avg_items_per_order,
       MEDIAN(items_per_order)         AS median_items_per_order,
       MAX(items_per_order)            AS max_items_per_order
FROM order_item_counts"""

In [8]:
q1_card_result = pd.read_sql(q1_card, engine)
q1_card_result

,orders_with_items,avg_items_per_order,median_items_per_order,max_items_per_order
0,98666,1.142,1,21


### q2

In [9]:
q2_base = """
CREATE OR REPLACE VIEW v_analytical_base AS
WITH latest_review AS (
    SELECT order_id, review_score
    FROM (
        SELECT order_id,
               review_score,
               ROW_NUMBER() OVER (
                   PARTITION BY order_id
                   ORDER BY review_answer_timestamp DESC, review_id
               ) AS rn
        FROM olist_order_reviews
    )
    WHERE rn = 1
)
SELECT
    o.order_id,
    o.customer_id,
    TRUNC(o.order_purchase_timestamp)       AS order_purchase_date,
    TRUNC(o.order_delivered_customer_date)  AS order_delivered_date,
    TRUNC(o.order_delivered_customer_date)
      - TRUNC(o.order_purchase_timestamp)   AS delivery_days,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.price,
    oi.freight_value,
    COALESCE(nt.name_english,
             p.product_category_name,
             'unknown')                     AS category_name,
    s.seller_city,
    s.seller_state,
    lr.review_score
FROM olist_orders o
JOIN olist_order_items oi ON oi.order_id = o.order_id
LEFT JOIN olist_products p ON p.product_id = oi.product_id
LEFT JOIN name_translation nt ON nt.name = p.product_category_name
LEFT JOIN olist_sellers s ON s.seller_id = oi.seller_id
LEFT JOIN latest_review lr ON lr.order_id = o.order_id
WHERE o.order_status = 'delivered'"""

q2_order = """
CREATE OR REPLACE VIEW v_order_base AS
SELECT order_id,
       MIN(order_purchase_date) AS order_purchase_date,
       SUM(price)               AS order_revenue,
       MIN(delivery_days)       AS delivery_days,
       MIN(review_score)        AS review_score
FROM v_analytical_base
GROUP BY order_id"""

q2_category = """
CREATE OR REPLACE VIEW v_order_category AS
SELECT order_id,
       category_name,
       SUM(price)         AS category_revenue,
       MIN(delivery_days) AS delivery_days,
       MIN(review_score)  AS review_score
FROM v_analytical_base
GROUP BY order_id, category_name"""

In [10]:
with engine.begin() as conn:
    for view_sql in (q2_base, q2_order, q2_category):
        conn.execute(text(view_sql))

In [11]:
q2_check = """
SELECT 'v_analytical_base' AS view_name, COUNT(*) AS row_count FROM v_analytical_base
UNION ALL SELECT 'v_order_base', COUNT(*) FROM v_order_base
UNION ALL SELECT 'v_order_category', COUNT(*) FROM v_order_category"""

q2_check_result = pd.read_sql(q2_check, engine)
q2_check_result

,view_name,row_count
0,v_analytical_base,110197
1,v_order_base,96478
2,v_order_category,97276


### q3

In [12]:
q3 = """
WITH direct_rev AS (
    SELECT SUM(oi.price) AS revenue_direct
    FROM olist_order_items oi
    JOIN olist_orders o ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
),
naive_rev AS (
    SELECT SUM(oi.price) AS revenue_naive
    FROM olist_orders o
    JOIN olist_order_items oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products p ON p.product_id = oi.product_id
    LEFT JOIN name_translation nt ON nt.name = p.product_category_name
    LEFT JOIN olist_sellers s ON s.seller_id = oi.seller_id
    LEFT JOIN olist_order_reviews r ON r.order_id = o.order_id
    WHERE o.order_status = 'delivered'
),
fixed_rev AS (
    SELECT SUM(price) AS revenue_fixed FROM v_analytical_base
)
SELECT d.revenue_direct,
       n.revenue_naive,
       f.revenue_fixed,
       n.revenue_naive - d.revenue_direct AS inflation_amount,
       ROUND((n.revenue_naive - d.revenue_direct) / d.revenue_direct * 100, 2) AS inflation_pct,
       CASE WHEN f.revenue_fixed = d.revenue_direct THEN 'PASS' ELSE 'FAIL' END AS fixed_status
FROM direct_rev d
CROSS JOIN naive_rev n
CROSS JOIN fixed_rev f"""

In [13]:
q3_result = pd.read_sql(q3, engine)
q3_result

,revenue_direct,revenue_naive,revenue_fixed,inflation_amount,inflation_pct,fixed_status
0,13221498.11,16691714.62,13221498.11,3470216.51,26.25,PASS


In [14]:
q3_steps = """
SELECT 'items only' AS step, COUNT(*) AS row_count, SUM(oi.price) AS revenue
FROM olist_orders o
JOIN olist_order_items oi ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
UNION ALL
SELECT '+ products', COUNT(*), SUM(oi.price)
FROM olist_orders o
JOIN olist_order_items oi ON oi.order_id = o.order_id
LEFT JOIN olist_products p ON p.product_id = oi.product_id
WHERE o.order_status = 'delivered'
UNION ALL
SELECT '+ translation', COUNT(*), SUM(oi.price)
FROM olist_orders o
JOIN olist_order_items oi ON oi.order_id = o.order_id
LEFT JOIN olist_products p ON p.product_id = oi.product_id
LEFT JOIN name_translation nt ON nt.name = p.product_category_name
WHERE o.order_status = 'delivered'
UNION ALL
SELECT '+ reviews (raw)', COUNT(*), SUM(oi.price)
FROM olist_orders o
JOIN olist_order_items oi ON oi.order_id = o.order_id
LEFT JOIN olist_products p ON p.product_id = oi.product_id
LEFT JOIN name_translation nt ON nt.name = p.product_category_name
LEFT JOIN olist_order_reviews r ON r.order_id = o.order_id
WHERE o.order_status = 'delivered'"""

In [15]:
q3_steps_result = pd.read_sql(q3_steps, engine)
q3_steps_result["row_delta"] = q3_steps_result["row_count"].diff()
q3_steps_result["revenue_delta"] = q3_steps_result["revenue"].diff()
q3_steps_result

,step,row_count,revenue,row_delta,revenue_delta
0,items only,110197,13221498.11,NaN,NaN
1,+ products,110197,13221498.11,0.0,0.00
2,+ translation,110197,13221498.11,0.0,0.00
3,+ reviews (raw),139037,16691714.62,28840.0,3470216.51


### q4

In [16]:
q4 = """
SELECT
    ROUND(SUM(order_revenue), 2)             AS total_revenue,
    ROUND(SUM(order_revenue) / COUNT(*), 2)  AS aov,
    ROUND(AVG(delivery_days), 2)             AS avg_delivery_days,
    ROUND(AVG(review_score), 2)              AS avg_review_score
FROM v_order_base"""

In [18]:
q4_result = pd.read_sql(q4, engine)
q4_result

DatabaseError: Execution failed on sql '
SELECT
    ROUND(SUM(order_revenue), 2)             AS total_revenue,
    ROUND(SUM(order_revenue) / COUNT(*), 2)  AS aov,
    ROUND(AVG(delivery_days), 2)             AS avg_delivery_days,
    ROUND(AVG(review_score), 2)              AS avg_review_score
FROM v_order_base': (oracledb.exceptions.DatabaseError) ORA-01722: invalid number
Help: https://docs.oracle.com/error-help/db/ora-01722/
[SQL: 
SELECT
    ROUND(SUM(order_revenue), 2)             AS total_revenue,
    ROUND(SUM(order_revenue) / COUNT(*), 2)  AS aov,
    ROUND(AVG(delivery_days), 2)             AS avg_delivery_days,
    ROUND(AVG(review_score), 2)              AS avg_review_score
FROM v_order_base]
(Background on this error at: https://sqlalche.me/e/21/4xp6)

### q5

In [ ]:
q5 = """
SELECT
    category_name,
    COUNT(DISTINCT order_id)                                   AS order_count,
    ROUND(SUM(category_revenue), 2)                            AS total_revenue,
    ROUND(SUM(category_revenue) / COUNT(DISTINCT order_id), 2) AS aov,
    ROUND(AVG(delivery_days), 2)                               AS avg_delivery_days,
    ROUND(AVG(review_score), 2)                                AS avg_review_score
FROM v_order_category
GROUP BY category_name
ORDER BY total_revenue DESC"""

In [ ]:
q5_result = pd.read_sql(q5, engine)
q5_result

### q6

In [ ]:
q6 = """
SELECT
    TO_CHAR(order_purchase_date, 'YYYY-MM')  AS month_id,
    COUNT(*)                                 AS order_count,
    ROUND(SUM(order_revenue), 2)             AS total_revenue,
    ROUND(SUM(order_revenue) / COUNT(*), 2)  AS aov,
    ROUND(AVG(delivery_days), 2)             AS avg_delivery_days,
    ROUND(AVG(review_score), 2)              AS avg_review_score
FROM v_order_base
GROUP BY TO_CHAR(order_purchase_date, 'YYYY-MM')
ORDER BY month_id"""

In [ ]:
q6_result = pd.read_sql(q6, engine)
q6_result

### q7

In [ ]:
q7 = """
WITH category_revenue AS (
    SELECT category_name, SUM(price) AS total_revenue
    FROM v_analytical_base
    GROUP BY category_name
),
ranked AS (
    SELECT category_name,
           total_revenue,
           ROW_NUMBER() OVER (ORDER BY total_revenue DESC, category_name) AS rn_top,
           ROW_NUMBER() OVER (ORDER BY total_revenue ASC,  category_name) AS rn_bottom
    FROM category_revenue
)
SELECT category_name,
       ROUND(total_revenue, 2) AS total_revenue,
       CASE WHEN rn_top <= 5 THEN 'TOP_5' ELSE 'BOTTOM_5' END AS bucket
FROM ranked
WHERE rn_top <= 5 OR rn_bottom <= 5
ORDER BY CASE WHEN rn_top <= 5 THEN 0 ELSE 1 END, total_revenue DESC"""

In [ ]:
q7_result = pd.read_sql(q7, engine)
q7_result

### q8

In [ ]:
q8 = """
WITH category_review AS (
    SELECT category_name,
           COUNT(DISTINCT order_id) AS order_count,
           AVG(review_score)        AS avg_score
    FROM v_order_category
    GROUP BY category_name
    HAVING COUNT(DISTINCT order_id) >= 100
),
ranked AS (
    SELECT category_name,
           order_count,
           avg_score,
           ROW_NUMBER() OVER (ORDER BY avg_score DESC, category_name) AS rn_top,
           ROW_NUMBER() OVER (ORDER BY avg_score ASC,  category_name) AS rn_bottom
    FROM category_review
)
SELECT category_name,
       order_count,
       ROUND(avg_score, 2) AS avg_review_score,
       CASE WHEN rn_top <= 5 THEN 'TOP_5' ELSE 'BOTTOM_5' END AS bucket
FROM ranked
WHERE rn_top <= 5 OR rn_bottom <= 5
ORDER BY CASE WHEN rn_top <= 5 THEN 0 ELSE 1 END, avg_score DESC"""

In [ ]:
q8_result = pd.read_sql(q8, engine)
q8_result

### q9

In [ ]:
q9 = """
WITH seller_order AS (
    SELECT seller_id,
           order_id,
           MIN(seller_city)   AS seller_city,
           MIN(seller_state)  AS seller_state,
           SUM(price)         AS revenue,
           MIN(delivery_days) AS delivery_days,
           MIN(review_score)  AS review_score
    FROM v_analytical_base
    GROUP BY seller_id, order_id
)
SELECT
    seller_id,
    MIN(seller_city)                          AS seller_city,
    MIN(seller_state)                         AS seller_state,
    COUNT(*)                                  AS order_count,
    ROUND(SUM(revenue), 2)                    AS total_revenue,
    ROUND(SUM(revenue) / COUNT(*), 2)         AS aov,
    ROUND(AVG(delivery_days), 2)              AS avg_delivery_days,
    ROUND(AVG(review_score), 2)               AS avg_review_score,
    RANK() OVER (ORDER BY SUM(revenue) DESC)  AS revenue_rank
FROM seller_order
GROUP BY seller_id
ORDER BY revenue_rank"""

In [ ]:
q9_result = pd.read_sql(q9, engine)
q9_result.head(10)

In [ ]:
top10_share_pct = q9_result.nsmallest(10, "revenue_rank")["total_revenue"].sum() / q9_result["total_revenue"].sum() * 100
print(f"Top 10 sellers = {top10_share_pct:.1f}% of total revenue")

### q10

In [ ]:
q10 = """
SELECT
    category_name,
    COUNT(DISTINCT order_id)         AS order_count,
    ROUND(AVG(delivery_days), 2)     AS avg_delivery_days,
    ROUND(AVG(review_score), 2)      AS avg_review_score
FROM v_order_category
GROUP BY category_name
HAVING COUNT(DISTINCT order_id) >= 100
ORDER BY avg_delivery_days"""

In [ ]:
q10_result = pd.read_sql(q10, engine)
q10_result

In [ ]:
pearson_r = q10_result["avg_delivery_days"].corr(q10_result["avg_review_score"], method="pearson")
spearman_rho = q10_result["avg_delivery_days"].corr(q10_result["avg_review_score"], method="spearman")
print(f"Pearson r = {pearson_r:.3f} | Spearman rho = {spearman_rho:.3f} | n = {len(q10_result)} categories")

Correlation ≠ causation. This is a category-level correlation (each point is a category average), not order-level. Category mix, price level and seller location can drive both delivery time and ratings.

## Charts

### monthly revenue trend

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(q6_result["month_id"], q6_result["total_revenue"] / 1e6, marker="o", color="tab:blue")
ax.set_title("Monthly Revenue Trend (delivered orders)")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (million BRL)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### top / bottom 5 categories by revenue

In [ ]:
q7_sorted = q7_result.sort_values("total_revenue")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(q7_sorted["category_name"], q7_sorted["total_revenue"],
        color=q7_sorted["bucket"].map({"TOP_5": "tab:green", "BOTTOM_5": "tab:red"}))
ax.set_title("Top 5 vs Bottom 5 Categories by Revenue")
ax.set_xlabel("Revenue (BRL)")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()

### top / bottom 5 categories by review score

In [ ]:
q8_sorted = q8_result.sort_values("avg_review_score")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(q8_sorted["category_name"], q8_sorted["avg_review_score"],
        color=q8_sorted["bucket"].map({"TOP_5": "tab:green", "BOTTOM_5": "tab:red"}))
ax.set_title("Top 5 vs Bottom 5 Categories by Avg Review Score (min 100 orders)")
ax.set_xlabel("Average review score (1-5)")
ax.set_ylabel("Category")
plt.tight_layout()
plt.show()

### delivery days vs review score

In [ ]:
x = q10_result["avg_delivery_days"].astype(float).to_numpy()
y = q10_result["avg_review_score"].astype(float).to_numpy()
slope, intercept = np.polyfit(x, y, deg=1)
grid = np.linspace(x.min(), x.max(), 100)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x, y, s=q10_result["order_count"] / q10_result["order_count"].max() * 600 + 30,
           alpha=0.6, color="tab:blue", edgecolor="white")
ax.plot(grid, slope * grid + intercept, color="tab:red", label=f"Trend line (r = {pearson_r:.2f})")
ax.set_title("Delivery Time vs Customer Satisfaction by Category (bubble size = order count)")
ax.set_xlabel("Average delivery days")
ax.set_ylabel("Average review score (1-5)")
ax.legend()
plt.tight_layout()
plt.show()

## Operational insights

**Insight 1 — Revenue concentration by category** *(Q5, Q7)*
`bed_bath_table` is the top category: 1.71M BRL, about 12.6% of the 13.59M BRL total (2016–2018, delivered orders). Top-5 combined share is [Q7 top-5 cəmi / 13.59M]%. Dependence on a few categories is a risk: a stock-out, price war or seller loss in one of them moves total revenue noticeably. Action: set category-level revenue monitoring with alerts, and grow mid-tier categories through targeted promotions.

**Insight 2 — Delivery time and satisfaction** *(Q10)*
Delivery delay (estimated vs actual) and review score show r = -0.43: later delivery goes with lower ratings. Categories with delivery above [Q10 median] days average [___] review score vs [___] for faster ones. This is a correlation, not proof of cause: category mix, price level and seller location can drive both. Action: set a maximum delivery-time SLA per category, flag sellers who breach it, and send proactive delay notifications to customers.

**Insight 3 — Weak categories by review score** *(Q8)*
Bottom 5 (min 100 orders): [Q8 bottom-5 adları], scores [___] vs overall 4.09. Action: for each weak category, separate product quality complaints from logistics complaints (review text, delivery days) before acting; a delivery problem and a product problem need different owners. Note: `security_and_services` (2.5) is excluded by the 100-order threshold if its volume is small, so do not cite it as a Q8 result.

**Insight 4 — Seller concentration** *(Q9)*
Top 10 sellers generate 12.7% of total revenue. This is moderate concentration: no single seller is critical, but the top group still matters. Action: protect the top sellers with retention programs and SLA monitoring, and check that their delivery days and review scores are not below the platform averages (12.1 days, 4.09).

**Insight 5 — Monthly trend** *(Q6)*
Revenue peaked in [Q6 pik ay] ([___] BRL); average delivery days in that month were [___] vs 12.1 overall. If delivery days rise in peak months, capacity is the constraint. Action: plan logistics capacity and seller stock ahead of peak months.